# Notebook 01 — Model comparison and Hb range

**Project:** conjunctiva photo → hemoglobin (Hb) range → anaemia screening (WHO 2024 cutoffs)

**What this notebook does**
1. Loads the cleaned data (CP-AnemiC from Ghana, Eyes-Defy-Anemia from India and Italy).
2. Compares many models on the same fixed tests, so the comparison is fair.
3. Tunes the winner.
4. Turns the Hb prediction into a **range** with a stated confidence, and a three-way answer:
   *anaemic / not anaemic / uncertain*.
5. Saves every table, plot and the final model to `/kaggle/working`.

**How to run on Kaggle:** add the private dataset (`anemia-data`), set *Accelerator = GPU T4*,
set *Internet = On* (needed once, to download the pretrained networks), then *Run All*.

In [ ]:
import os, glob, json, time, warnings, random
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, KFold, GridSearchCV
from sklearn.metrics import roc_auc_score, roc_curve
import xgboost as xgb
import lightgbm as lgb
import joblib

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Two switches used only for a quick self-test of the notebook. Leave them alone on Kaggle.
QUICK = os.environ.get("ANEMIA_QUICK", "0") == "1"            # fewer repeats / epochs
PRETRAINED = os.environ.get("ANEMIA_PRETRAINED", "1") == "1"  # download ImageNet weights

def find_data():
    hits = glob.glob("/kaggle/input/**/features.csv", recursive=True)
    if os.environ.get("ANEMIA_DATA"):
        hits = [os.path.join(os.environ["ANEMIA_DATA"], "features.csv")] + hits
    assert hits, "features.csv not found — add the anemia-data dataset to this notebook"
    return os.path.dirname(hits[0])

DATA = find_data()
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("ANEMIA_OUT", "outputs")
os.makedirs(OUT, exist_ok=True)
print("data:", DATA, "| outputs:", OUT, "| quick:", QUICK, "| pretrained:", PRETRAINED)

## 1. Data

One row per subject. `anemic = 1` when the lab Hb is below the WHO 2024 cutoff for that
person's age and sex. `usable = 0` marks the CP-AnemiC images that turned out to be copies of
each other with different labels; they are left out everywhere.

In [ ]:
df_all = pd.read_csv(os.path.join(DATA, "features.csv"))
df = df_all[(df_all.usable == 1) & df_all.hb.notna()].reset_index(drop=True)
print(df_all.groupby("site").agg(total=("hb", "size"), usable=("usable", "sum")))
print()
print(df.groupby("site").agg(n=("hb", "size"), anaemic=("anemic", "sum"),
                             hb_mean=("hb", "mean"), hb_min=("hb", "min"), hb_max=("hb", "max")).round(1))

y = df.anemic.values.astype(int)          # class label
hb = df.hb.values.astype(float)           # lab hemoglobin, g/dL
cut = df.who_cutoff.values.astype(float)  # WHO 2024 cutoff for this person
site = df.site.values
CP = np.where(df.dataset == "cp_anemic")[0]
ED = np.where(df.dataset == "eyes_defy")[0]

## 2. The tests every model must pass

| Test | Train on | Test on | What it tells us |
|---|---|---|---|
| Within CP-AnemiC | 4/5 of Ghana | the other 1/5, rotated (5-fold CV, repeated) | best case: same camera, same population |
| Within Eyes-Defy | 4/5 of India+Italy | the other 1/5, rotated | same, reported for India and Italy **separately** |
| CP → ED | all Ghana children | India and Italy adults | does it survive a new population and camera? |
| ED → CP | all India+Italy adults | Ghana children | the reverse direction |

The splits are created once and reused for every model.

**Two ways to get the anaemia answer**
- **Route A (classify):** the model outputs "probability of anaemia" directly.
- **Route B (estimate Hb, then compare):** the model outputs Hb in g/dL, and we compare it with
  the person's own WHO cutoff. Age and sex are typed in by the user at screening time, so using
  the cutoff is legitimate. Route B is what the final system needs, because it also gives the Hb range.

**Scores:** AUC (0.5 = coin toss, 1.0 = perfect ranking), sensitivity (anaemic people caught),
specificity (healthy people cleared), MAE (average Hb error in g/dL).

In [ ]:
N_REP = 1 if QUICK else 5
def make_splits():
    s = {}
    rs = RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REP, random_state=SEED)
    s["within_CP"] = [(CP[a], CP[b]) for a, b in rs.split(CP, y[CP])]
    strat = np.array([f"{a}_{b}" for a, b in zip(site[ED], y[ED])])   # keep site and class balanced
    s["within_ED"] = [(ED[a], ED[b]) for a, b in rs.split(ED, strat)]
    s["CP_to_ED"] = [(CP, ED)]
    s["ED_to_CP"] = [(ED, CP)]
    return s
SPLITS = make_splits()
SUBSETS = {"within_CP": ["Ghana"], "within_ED": ["India", "Italy", "India+Italy"],
           "CP_to_ED": ["India", "Italy", "India+Italy"], "ED_to_CP": ["Ghana"]}

def _auc(t, s):
    return roc_auc_score(t, s) if len(np.unique(t)) == 2 else np.nan

def metrics(idx, score, pred, hb_pred=None):
    t = y[idx]
    m = {"n": len(idx), "auc": _auc(t, score),
         "sens": (pred[t == 1] == 1).mean() if (t == 1).any() else np.nan,
         "spec": (pred[t == 0] == 0).mean() if (t == 0).any() else np.nan}
    m["bal_acc"] = np.nanmean([m["sens"], m["spec"]])
    if hb_pred is not None:
        m["mae"] = np.abs(hb_pred - hb[idx]).mean()
        m["r"] = np.corrcoef(hb_pred, hb[idx])[0, 1]
    return m

def run_model(est, X, route, tr, te):
    """Fit on tr, return (score, predicted class, predicted Hb or None) for te."""
    est = clone(est)
    if route == "clf":
        est.fit(X[tr], y[tr])
        if hasattr(est, "predict_proba"):
            sc = est.predict_proba(X[te])[:, 1]; pred = (sc >= 0.5).astype(int)
        else:
            sc = est.decision_function(X[te]); pred = (sc >= 0).astype(int)
        return sc, pred, None
    est.fit(X[tr], hb[tr])
    hp = est.predict(X[te])
    return cut[te] - hp, (hp < cut[te]).astype(int), hp     # score: how far below the cutoff

def evaluate(est, X, route, feature_set, model, keep=None, max_rep=None):
    """Run all four tests. Returns result rows; optionally keeps out-of-sample predictions."""
    rows = []
    for test, folds in SPLITS.items():
        n_rep = max(1, len(folds) // 5) if test.startswith("within") else 1
        if max_rep: n_rep = min(n_rep, max_rep)
        per_rep = {s: [] for s in SUBSETS[test]}
        for rep in range(n_rep):
            fs = folds[rep * 5:(rep + 1) * 5] if test.startswith("within") else folds
            idx = np.concatenate([te for _, te in fs])
            sc = np.zeros(len(idx)); pr = np.zeros(len(idx), int); hp = np.zeros(len(idx)); pos = 0
            for tr, te in fs:
                a, b, c = run_model(est, X, route, tr, te)
                sc[pos:pos + len(te)] = a; pr[pos:pos + len(te)] = b
                if c is not None: hp[pos:pos + len(te)] = c
                pos += len(te)
            if keep is not None and rep == 0:
                keep[test] = dict(idx=idx, score=sc, pred=pr, hb_pred=hp if route == "reg" else None)
            for s in SUBSETS[test]:
                msk = np.ones(len(idx), bool) if "+" in s or s == "Ghana" else (site[idx] == s)
                per_rep[s].append(metrics(idx[msk], sc[msk], pr[msk], hp[msk] if route == "reg" else None))
        for s, ms in per_rep.items():
            r = {"feature_set": feature_set, "model": model, "route": route, "test": test, "subset": s}
            for k in ms[0]:
                r[k] = float(np.nanmean([m[k] for m in ms]))
            r["auc_sd"] = float(np.nanstd([m["auc"] for m in ms]))
            rows.append(r)
    return rows

## 3. Inputs the models can use (feature sets)

- **colour_ratio** — 13 numbers describing the *redness* of the conjunctiva in ways that cancel out
  brightness (for example log(R/G), the a\* channel of CIELAB). No colour card needed.
- **colour_all** — the ratio features plus raw brightness-dependent colour (mean R, G, B…).
  Raw colour depends on the camera and lighting, so expect it to help within a dataset and hurt across.
- **deep embeddings** — each cut-out is passed through a network pretrained on ImageNet
  (ResNet-50, EfficientNet-B0, ConvNeXt-Tiny) and we keep its internal description of the image
  (about 1,000 numbers). The network is *not* trained here; it is used as a fixed feature extractor.
- **colour_ratio + best embedding** — both together.

In [ ]:
RATIO = [c for c in df.columns if c.startswith("ratio_")]
RAW = [c for c in df.columns if c.startswith("raw_")]
FEATS = {"colour_ratio": df[RATIO].values.astype(np.float32),
         "colour_all": df[RATIO + RAW].values.astype(np.float32)}
print({k: v.shape for k, v in FEATS.items()})

In [ ]:
import torch, torch.nn as nn, torchvision
from torchvision import models as tvm
DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
print("torch", torch.__version__, "| device:", DEV)
if DEV == "cpu" and not QUICK:
    print("WARNING: no GPU. Turn on Accelerator = GPU T4 in the notebook settings; the CNN part will be slow.")

IMG = 224
BG = np.array([124, 116, 104], np.float32)   # ImageNet mean colour: a neutral background for the cut-out
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def load_square(path, size=IMG):
    """Cut-out (RGBA) → square RGB image on a neutral background, uint8."""
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if im.ndim == 2: im = cv2.cvtColor(im, cv2.COLOR_GRAY2BGRA)
    if im.shape[2] == 3: im = np.dstack([im, np.full(im.shape[:2], 255, np.uint8)])
    rgb = im[:, :, 2::-1].astype(np.float32); a = im[:, :, 3:].astype(np.float32) / 255.0
    comp = rgb * a + BG * (1 - a)
    h, w = comp.shape[:2]; s = max(h, w)
    sq = np.tile(BG, (s, s, 1)); y0, x0 = (s - h) // 2, (s - w) // 2
    sq[y0:y0 + h, x0:x0 + w] = comp
    return cv2.resize(sq, (size, size), interpolation=cv2.INTER_AREA).clip(0, 255).astype(np.uint8)

t0 = time.time()
IMGS = np.stack([load_square(os.path.join(DATA, p)) for p in df.conj_path])
print("images:", IMGS.shape, f"{time.time() - t0:.0f}s")

fig, ax = plt.subplots(2, 6, figsize=(13, 4.6))
order = [CP[np.argsort(hb[CP])][i] for i in np.linspace(0, len(CP) - 1, 6).astype(int)] + \
        [ED[np.argsort(hb[ED])][i] for i in np.linspace(0, len(ED) - 1, 6).astype(int)]
for a, i in zip(ax.ravel(), order):
    a.imshow(IMGS[i]); a.set_title(f"{site[i]}  Hb {hb[i]:.1f}", fontsize=9); a.axis("off")
plt.suptitle("What the networks see (low Hb on the left, high Hb on the right)"); plt.tight_layout()
plt.savefig(os.path.join(OUT, "01_inputs.png"), dpi=120); plt.show()

In [ ]:
def backbone(name):
    w = "DEFAULT" if PRETRAINED else None
    if name == "resnet50":
        m = tvm.resnet50(weights=w); m.fc = nn.Identity()
    elif name == "efficientnet_b0":
        m = tvm.efficientnet_b0(weights=w); m.classifier = nn.Identity()
    elif name == "convnext_tiny":
        m = tvm.convnext_tiny(weights=w); m.classifier = nn.Sequential(nn.Flatten(1))
    return m

def to_tensor(batch_uint8):
    x = torch.from_numpy(batch_uint8).permute(0, 3, 1, 2).float() / 255.0
    return (x - MEAN) / STD

@torch.no_grad()
def embed(name, bs=32):
    m = backbone(name).to(DEV).eval(); out = []
    for i in range(0, len(IMGS), bs):
        x = to_tensor(IMGS[i:i + bs]).to(DEV)
        e = (m(x) + m(torch.flip(x, dims=[3]))) / 2      # average the image and its mirror
        out.append(e.float().cpu().numpy())
    del m
    return np.concatenate(out)

BACKBONES = ["efficientnet_b0"] if QUICK else ["resnet50", "efficientnet_b0", "convnext_tiny"]
for b in BACKBONES:
    t0 = time.time(); FEATS["emb_" + b] = embed(b)
    print(b, FEATS["emb_" + b].shape, f"{time.time() - t0:.0f}s")
np.savez_compressed(os.path.join(OUT, "embeddings.npz"), subject_id=df.subject_id.values,
                    **{k: v for k, v in FEATS.items() if k.startswith("emb_")})

## 4. The models

| Name | Idea in one line |
|---|---|
| Logistic / Ridge | a weighted sum of the features — simple, hard to overfit |
| SVM | draws a curved boundary — good with few samples |
| Random forest | many decision trees voting |
| XGBoost, LightGBM | trees built one after another, each fixing the previous one's mistakes |

Each comes as a classifier (Route A) and a regressor (Route B). Features are standardised first.
The deep embeddings have about 1,000 numbers but we have only a few hundred images, so they are
first squeezed to 32 numbers with PCA to stop the models memorising noise.

In [ ]:
NJ = -1
def zoo(route):
    if route == "clf":
        return {"logistic": LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000),
                "svm_rbf": SVC(C=1.0, kernel="rbf", class_weight="balanced"),
                "random_forest": RandomForestClassifier(400, min_samples_leaf=3, class_weight="balanced",
                                                        n_jobs=NJ, random_state=SEED),
                "xgboost": xgb.XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8,
                                             colsample_bytree=0.8, n_jobs=4, random_state=SEED, verbosity=0),
                "lightgbm": lgb.LGBMClassifier(n_estimators=300, num_leaves=7, learning_rate=0.05, subsample=0.8,
                                               subsample_freq=1, colsample_bytree=0.8, min_child_samples=10,
                                               class_weight="balanced", n_jobs=4, random_state=SEED, verbose=-1)}
    return {"ridge": Ridge(alpha=10.0),
            "svr_rbf": SVR(C=3.0, epsilon=0.3),
            "random_forest": RandomForestRegressor(400, min_samples_leaf=3, n_jobs=NJ, random_state=SEED),
            "xgboost": xgb.XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8,
                                        colsample_bytree=0.8, n_jobs=4, random_state=SEED, verbosity=0),
            "lightgbm": lgb.LGBMRegressor(n_estimators=300, num_leaves=7, learning_rate=0.05, subsample=0.8,
                                          subsample_freq=1, colsample_bytree=0.8, min_child_samples=10,
                                          n_jobs=4, random_state=SEED, verbose=-1)}

def pipe(model, n_features):
    steps = [("sc", StandardScaler())]
    if n_features > 100:
        steps += [("pca", PCA(n_components=32, random_state=SEED)), ("sc2", StandardScaler())]
    return Pipeline(steps + [("m", model)])

RESULTS = []
def run_feature_set(fs):
    X = FEATS[fs]
    for route in ("clf", "reg"):
        for name, mdl in zoo(route).items():
            t0 = time.time()
            RESULTS.extend(evaluate(pipe(mdl, X.shape[1]), X, route, fs, name))
            print(f"  {fs:28s} {route} {name:14s} {time.time() - t0:5.0f}s", flush=True)

for fs in list(FEATS):
    run_feature_set(fs)

### Ranking

One headline number per configuration: the **mean of six AUCs** — within Ghana, within India,
within Italy, Ghana→India, Ghana→Italy, and India+Italy→Ghana. A model has to do well on all six,
so it cannot win by being good on one dataset only. (India+Italy pooled is left out of the
headline: India is mostly anaemic and Italy mostly not, so a pooled score partly rewards
recognising the country.)

In [ ]:
HEAD = [("within_CP", "Ghana"), ("within_ED", "India"), ("within_ED", "Italy"),
        ("CP_to_ED", "India"), ("CP_to_ED", "Italy"), ("ED_to_CP", "Ghana")]
def leaderboard(results):
    r = pd.DataFrame(results)
    r["col"] = r.test + ":" + r.subset
    cfg = ["feature_set", "model", "route"]
    wide = r.pivot_table(index=cfg, columns="col", values="auc")
    cols = [a + ":" + b for a, b in HEAD]
    wide = wide[cols]
    wide.insert(0, "headline_auc", wide.mean(axis=1))
    wide.insert(1, "within_mean", wide[cols[:3]].mean(axis=1))
    wide.insert(2, "cross_mean", wide[cols[3:]].mean(axis=1))
    mae = r[r.route == "reg"].pivot_table(index=cfg, columns="col", values="mae")
    if len(mae):
        wide["mae_within_CP"] = mae["within_CP:Ghana"]
        wide["mae_within_ED"] = mae["within_ED:India+Italy"]
        wide["mae_CP_to_ED"] = mae["CP_to_ED:India+Italy"]
        wide["mae_ED_to_CP"] = mae["ED_to_CP:Ghana"]
    return wide.sort_values("headline_auc", ascending=False)

def best_embedding(results):
    lb = leaderboard(results).reset_index()
    lb = lb[lb.feature_set.str.startswith("emb_")]
    return lb.groupby("feature_set").headline_auc.max().idxmax()

BEST_EMB = best_embedding(RESULTS)
print("best embedding:", BEST_EMB)
# colour + embedding together: PCA the embedding to 32 numbers first is done inside the pipeline for
# the embedding alone; here we concatenate colour with a fixed 16-number PCA summary fitted without labels.
emb = FEATS[BEST_EMB]
emb_small = PCA(n_components=16, random_state=SEED).fit_transform(StandardScaler().fit_transform(emb))
FEATS["colour_ratio+emb"] = np.hstack([FEATS["colour_ratio"], emb_small]).astype(np.float32)
run_feature_set("colour_ratio+emb")

LB = leaderboard(RESULTS)
pd.DataFrame(RESULTS).to_csv(os.path.join(OUT, "01_all_results.csv"), index=False)
LB.round(3).to_csv(os.path.join(OUT, "01_leaderboard.csv"))
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(LB.round(3).head(15).to_string())

Note on `colour_ratio+emb`: the 16-number PCA summary of the embedding is computed on all images
at once. PCA never sees the Hb labels, so this does not leak the answer, but it does see the test
images' appearance. Treat that row as slightly optimistic.

In [ ]:
top = LB.reset_index().head(12)
lab = top.feature_set + " | " + top.model + " | " + top.route
fig, ax = plt.subplots(figsize=(10, 5))
yy = np.arange(len(top))[::-1]
ax.barh(yy + 0.2, top.within_mean, 0.4, label="within-dataset (mean of 3)", color="#4c72b0")
ax.barh(yy - 0.2, top.cross_mean, 0.4, label="cross-dataset (mean of 3)", color="#dd8452")
ax.set_yticks(yy); ax.set_yticklabels(lab, fontsize=8); ax.axvline(0.5, color="k", lw=0.8, ls=":")
ax.set_xlim(0.4, 1.0); ax.set_xlabel("AUC"); ax.legend(loc="lower right"); ax.set_title("Top configurations")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "01_leaderboard.png"), dpi=130); plt.show()

## 5. A network trained end-to-end (fine-tuned CNN)

So far the networks were frozen. Here EfficientNet-B0 is **trained on our images** to predict Hb
directly (Route B). With only a few hundred images this can overfit, so:
- augmentation is limited to mirror, small rotation/zoom and small brightness changes —
  **never hue or saturation**, because colour is the signal;
- training is short, with a low learning rate and weight decay.

It goes through the same four tests (one repeat of the 5-fold splits, to keep the runtime down).

In [ ]:
from torchvision.transforms import v2 as T
HB_MU, HB_SD = 11.5, 2.0
EPOCHS = 1 if QUICK else 30
AUG = T.Compose([T.RandomHorizontalFlip(),
                 T.RandomAffine(degrees=12, translate=(0.05, 0.05), scale=(0.9, 1.1), fill=[124, 116, 104]),
                 T.ColorJitter(brightness=0.12)])

def train_cnn(tr, epochs=EPOCHS, seed=SEED):
    torch.manual_seed(seed)
    m = tvm.efficientnet_b0(weights="DEFAULT" if PRETRAINED else None)
    m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, 1))
    m = m.to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=2e-4, weight_decay=1e-2)
    steps = epochs * int(np.ceil(len(tr) / 32))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-4, total_steps=steps, pct_start=0.15)
    xs = torch.from_numpy(IMGS[tr]).permute(0, 3, 1, 2)                      # uint8, CPU
    ys = torch.tensor((hb[tr] - HB_MU) / HB_SD, dtype=torch.float32)
    lossf = nn.SmoothL1Loss()
    for ep in range(epochs):
        m.train(); perm = torch.randperm(len(tr))
        for i in range(0, len(tr), 32):
            b = perm[i:i + 32]
            if len(b) < 2: continue
            x = torch.stack([AUG(xs[j]) for j in b]).float() / 255.0
            x = ((x - MEAN) / STD).to(DEV)
            loss = lossf(m(x).squeeze(1), ys[b].to(DEV))
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return m

@torch.no_grad()
def predict_cnn(m, te):
    m.eval(); out = []
    for i in range(0, len(te), 64):
        x = to_tensor(IMGS[te[i:i + 64]]).to(DEV)
        out.append(((m(x) + m(torch.flip(x, dims=[3]))) / 2).squeeze(1).cpu().numpy())
    return np.concatenate(out) * HB_SD + HB_MU

CNN_KEEP = {}
def evaluate_cnn():
    rows = []
    for test, folds in SPLITS.items():
        fs = folds[:5] if test.startswith("within") else folds
        if QUICK: fs = fs[:1]
        idx = np.concatenate([te for _, te in fs]); hp = np.zeros(len(idx)); pos = 0
        for tr, te in fs:
            t0 = time.time(); m = train_cnn(tr); hp[pos:pos + len(te)] = predict_cnn(m, te); pos += len(te)
            del m; torch.cuda.empty_cache() if DEV == "cuda" else None
            print(f"  {test}: trained on {len(tr)}, tested on {len(te)}  {time.time() - t0:.0f}s", flush=True)
        sc = cut[idx] - hp; pr = (hp < cut[idx]).astype(int)
        CNN_KEEP[test] = dict(idx=idx, score=sc, pred=pr, hb_pred=hp)
        for s in SUBSETS[test]:
            msk = np.ones(len(idx), bool) if "+" in s or s == "Ghana" else (site[idx] == s)
            r = {"feature_set": "image (fine-tuned)", "model": "efficientnet_b0", "route": "reg",
                 "test": test, "subset": s, **metrics(idx[msk], sc[msk], pr[msk], hp[msk]), "auc_sd": 0.0}
            rows.append(r)
    return rows

RESULTS.extend(evaluate_cnn())
LB = leaderboard(RESULTS)
pd.DataFrame(RESULTS).to_csv(os.path.join(OUT, "01_all_results.csv"), index=False)
LB.round(3).to_csv(os.path.join(OUT, "01_leaderboard.csv"))
print(LB.round(3).head(15).to_string())
print("\nFine-tuned CNN rank:", list(LB.index.get_level_values(0)).index("image (fine-tuned)") + 1, "of", len(LB))

## 6. Tuning the winner

The best **Route B** configuration from the table (ignoring the fine-tuned CNN, which is tuned by
its training recipe) is tuned with a small grid search. The search runs **inside each training
fold only** ("nested" cross-validation), so the test images never influence the settings.
To keep the runtime down the tuned model is scored on one repeat of the splits.

Honest caveat: we chose the winner by looking at test scores of ~50 configurations, so its score
is a little optimistic. The cross-dataset numbers are the safer ones to quote.

In [ ]:
GRIDS = {"ridge": {"m__alpha": [0.1, 1, 10, 100, 1000]},
         "svr_rbf": {"m__C": [1, 3, 10], "m__epsilon": [0.1, 0.3, 0.6], "m__gamma": ["scale", 0.01]},
         "random_forest": {"m__min_samples_leaf": [1, 3, 6], "m__max_features": ["sqrt", 0.5, 1.0]},
         "xgboost": {"m__max_depth": [2, 3], "m__n_estimators": [150, 400], "m__learning_rate": [0.03, 0.1]},
         "lightgbm": {"m__num_leaves": [4, 8], "m__n_estimators": [150, 400], "m__learning_rate": [0.03, 0.1]}}

lb_reg = LB.reset_index()
lb_reg = lb_reg[(lb_reg.route == "reg") & (lb_reg.feature_set != "image (fine-tuned)")]
W = lb_reg.iloc[0]
print("winner (Route B):", W.feature_set, "|", W.model, "| headline AUC", round(W.headline_auc, 3))
XW = FEATS[W.feature_set]
base = pipe(zoo("reg")[W.model], XW.shape[1])
grid = GRIDS[W.model]
if QUICK: grid = {k: v[:2] for k, v in list(grid.items())[:1]}
tuned = GridSearchCV(base, grid, scoring="neg_mean_absolute_error",
                     cv=KFold(4, shuffle=True, random_state=SEED), n_jobs=1)
TUNED_KEEP = {}
t0 = time.time()
tuned_rows = evaluate(tuned, XW, "reg", W.feature_set, W.model + " (tuned)", keep=TUNED_KEEP, max_rep=1)
print(f"tuning took {time.time() - t0:.0f}s")
RESULTS.extend(tuned_rows)
LB = leaderboard(RESULTS)
cmp_ = LB.reset_index()
cmp_ = cmp_[(cmp_.feature_set == W.feature_set) & cmp_.model.isin([W.model, W.model + " (tuned)"]) & (cmp_.route == "reg")]
print(cmp_.round(3).to_string(index=False))
pd.DataFrame(RESULTS).to_csv(os.path.join(OUT, "01_all_results.csv"), index=False)
LB.round(3).to_csv(os.path.join(OUT, "01_leaderboard.csv"))

t_auc = cmp_[cmp_.model.str.endswith("(tuned)")].headline_auc.values[0]
FINAL_EST = tuned if t_auc >= W.headline_auc else base
FINAL_NAME = W.model + (" (tuned)" if t_auc >= W.headline_auc else "")
print("final model:", W.feature_set, "|", FINAL_NAME)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
for test, sub, c in [("within_CP", "Ghana", "#4c72b0"), ("within_ED", "India", "#55a868"), ("within_ED", "Italy", "#c44e52"),
                     ("CP_to_ED", "India", "#55a868"), ("CP_to_ED", "Italy", "#c44e52"), ("ED_to_CP", "Ghana", "#4c72b0")]:
    k = TUNED_KEEP[test]; msk = np.ones(len(k["idx"]), bool) if sub == "Ghana" else site[k["idx"]] == sub
    a = ax[0] if test.startswith("within") else ax[1]
    f, t, _ = roc_curve(y[k["idx"][msk]], k["score"][msk])
    a.plot(f, t, color=c, label=f"{sub} (AUC {_auc(y[k['idx'][msk]], k['score'][msk]):.2f})")
for a, ttl in zip(ax, ["Within dataset", "Cross dataset (tested on …)"]):
    a.plot([0, 1], [0, 1], "k:", lw=0.8); a.set_xlabel("1 − specificity"); a.set_ylabel("sensitivity")
    a.set_title(ttl); a.legend(loc="lower right")
plt.suptitle(f"ROC — {W.feature_set} | {W.model} (tuned), Route B"); plt.tight_layout()
plt.savefig(os.path.join(OUT, "01_roc_winner.png"), dpi=130); plt.show()

## 7. From one number to an Hb range

A single Hb value hides how unsure the model is. We use **conformal prediction**:
1. Inside the training data, predict each image with a model that did not see it and record the error.
2. Take the error size that covers, say, 80% of those cases — call it *q*.
3. For a new image, report **prediction ± q**. By construction this range should contain the true
   Hb about 80% of the time, *as long as the new images resemble the training ones*.

The screening answer then has three outcomes:
- whole range below the person's WHO cutoff → **likely anaemic**
- whole range above the cutoff → **likely not anaemic**
- range straddles the cutoff → **uncertain — recommend a blood test**

"Coverage" below is how often the true Hb really fell inside the range. Within a dataset it should
be close to the target. Across datasets it usually drops — that drop is itself a result worth reporting.

In [ ]:
def conformal_run(est, X, tr, te, alphas):
    """Return hb_pred for te and the half-width q for each alpha (from out-of-fold training errors)."""
    resid = np.zeros(len(tr))
    for a, b in KFold(5, shuffle=True, random_state=SEED).split(tr):
        e = clone(est).fit(X[tr[a]], hb[tr[a]]); resid[b] = np.abs(e.predict(X[tr[b]]) - hb[tr[b]])
    e = clone(est).fit(X[tr], hb[tr]); hp = e.predict(X[te])
    n = len(resid)
    qs = {al: float(np.quantile(resid, min(1.0, np.ceil((n + 1) * (1 - al)) / n))) for al in alphas}
    return hp, qs

ALPHAS = [0.2, 0.1]       # 80% and 90% ranges
def final_est_for(tr):
    """Tuned settings are chosen on the training rows only."""
    if isinstance(FINAL_EST, GridSearchCV):
        return clone(FINAL_EST).fit(XW[tr], hb[tr]).best_estimator_
    return FINAL_EST

crow, CONF_KEEP = [], {}
for test, folds in SPLITS.items():
    fs = folds[:5] if test.startswith("within") else folds
    idx = np.concatenate([te for _, te in fs]); hp = np.zeros(len(idx)); q = {al: np.zeros(len(idx)) for al in ALPHAS}; pos = 0
    for tr, te in fs:
        h, qs = conformal_run(final_est_for(tr), XW, tr, te, ALPHAS)
        hp[pos:pos + len(te)] = h
        for al in ALPHAS: q[al][pos:pos + len(te)] = qs[al]
        pos += len(te)
    CONF_KEEP[test] = dict(idx=idx, hb_pred=hp, q=q)
    for s in SUBSETS[test]:
        msk = np.ones(len(idx), bool) if "+" in s or s == "Ghana" else (site[idx] == s)
        i, h = idx[msk], hp[msk]
        for al in ALPHAS:
            lo, hi = h - q[al][msk], h + q[al][msk]
            call = np.where(hi < cut[i], 1, np.where(lo >= cut[i], 0, -1))     # 1 anaemic, 0 not, -1 uncertain
            dec = call >= 0
            crow.append({"test": test, "subset": s, "n": len(i), "target_coverage": 1 - al,
                         "actual_coverage": float(((hb[i] >= lo) & (hb[i] <= hi)).mean()),
                         "range_width_gdl": float(2 * q[al][msk].mean()),
                         "decided_frac": float(dec.mean()),
                         "accuracy_when_decided": float((call[dec] == y[i][dec]).mean()) if dec.any() else np.nan,
                         "anaemic_called": int((call == 1).sum()), "anaemic_called_correct": int(((call == 1) & (y[i] == 1)).sum()),
                         "not_anaemic_called": int((call == 0).sum()), "not_anaemic_called_correct": int(((call == 0) & (y[i] == 0)).sum()),
                         "mae": float(np.abs(h - hb[i]).mean())})
CONF = pd.DataFrame(crow)
CONF.round(3).to_csv(os.path.join(OUT, "02_hb_range.csv"), index=False)
print(CONF.round(2).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
for a, (test, sub) in zip(ax, [("within_CP", "Ghana"), ("within_ED", "India+Italy"), ("CP_to_ED", "India+Italy")]):
    k = CONF_KEEP[test]; i = k["idx"]; o = np.argsort(hb[i])
    a.fill_between(np.arange(len(i)), (k["hb_pred"] - k["q"][0.2])[o], (k["hb_pred"] + k["q"][0.2])[o],
                   color="#4c72b0", alpha=0.25, label="80% range")
    a.plot(np.arange(len(i)), k["hb_pred"][o], ".", ms=3, color="#4c72b0", label="predicted Hb")
    a.plot(np.arange(len(i)), hb[i][o], "-", color="k", lw=1.2, label="lab Hb")
    a.set_title(f"{test} ({sub})"); a.set_xlabel("subjects sorted by lab Hb"); a.set_ylabel("Hb (g/dL)")
ax[0].legend(); plt.tight_layout(); plt.savefig(os.path.join(OUT, "02_hb_range.png"), dpi=130); plt.show()

## 8. Final model

The chosen pipeline is refitted on **all** usable images from both datasets and saved together with
its range half-widths. This file is what the demo app will load.

If the winning feature set is `colour_ratio` or `colour_all`, the saved model works on the colour
features alone, so the phone-side code only needs the feature extraction from `src/features.py`.

In [ ]:
allidx = np.arange(len(df))
final = final_est_for(allidx)
_, q_all = conformal_run(final, XW, allidx, allidx[:1], ALPHAS)
final = clone(final).fit(XW, hb)
bundle = {"model": final, "feature_set": W.feature_set,
          "feature_names": (RATIO if W.feature_set == "colour_ratio" else RATIO + RAW if W.feature_set == "colour_all" else None),
          "half_width_80": q_all[0.2], "half_width_90": q_all[0.1],
          "trained_on": {"Ghana": int(len(CP)), "India+Italy": int(len(ED))}, "sklearn": __import__("sklearn").__version__}
joblib.dump(bundle, os.path.join(OUT, "final_hb_model.joblib"))
summary = {"winner_feature_set": W.feature_set, "winner_model": FINAL_NAME,
           "headline_auc": float(LB.reset_index().query("feature_set == @W.feature_set and model == @FINAL_NAME and route == 'reg'").headline_auc.values[0]),
           "half_width_80_gdl": q_all[0.2], "half_width_90_gdl": q_all[0.1],
           "best_embedding": BEST_EMB, "pretrained": PRETRAINED, "device": DEV, "n_repeats": N_REP, "cnn_epochs": EPOCHS}
json.dump(summary, open(os.path.join(OUT, "summary.json"), "w"), indent=2)
print(json.dumps(summary, indent=2))
print("\nfiles written:", sorted(os.listdir(OUT)))

## What to take away

- `01_leaderboard.csv` — every configuration ranked; quote the **cross-dataset** columns as the honest numbers.
- `02_hb_range.csv` — how wide the Hb range is, how often it contains the truth, and how often the
  system can commit to an answer.
- `final_hb_model.joblib` — the model for the demo.

Download the whole output folder (*Output → Download All*) and put it in
`MajorProject/results/kaggle_run_01/`.